# D15 · Introducción al NLP (Procesamiento de Lenguaje Natural)

**Formación Pública — Bootcamp de Ciencia de Datos para funcionarios públicos**
Bloque avanzado · IA aplicada (opcional) · Semana 18

---

## ¿Qué vas a lograr?

Hasta ahora tus datos fueron **números y categorías** ordenados en columnas. Pero buena parte de la información del Estado es **texto libre**: el nombre de una licitación, la descripción de un contrato, una pregunta en el foro de compras, un reclamo ciudadano. El **NLP** (Procesamiento de Lenguaje Natural) es la disciplina que enseña a la máquina a **leer y analizar texto a escala**.

En este módulo vas a tomar texto real de compras públicas y a recorrer la **tubería básica de todo proyecto de NLP**: limpiarlo, partirlo en palabras, contar lo que importa y construir un primer "clasificador" que adivina de qué trata un texto. Es el cimiento sobre el que después se montan los modelos de lenguaje (eso viene en D16).

Al terminar serás capaz de:

- Entender por qué el texto es un dato **difícil** y cómo se prepara.
- **Normalizar** texto en español (minúsculas, tildes, signos).
- **Tokenizar** y quitar *palabras vacías* (stopwords).
- Medir **frecuencias** y construir una **bolsa de palabras**.
- Programar un **clasificador simple por palabras clave**.

**Competencia de salida:** dado un texto real del Estado, prepararlo y extraer de él información cuantificable y útil.

**Entregable:** que las **4 celdas de chequeo** muestren ✅.

> **Requisito previo:** núcleo completo (M0–D13). Usamos solo la librería estándar de Python (`re`, `unicodedata`, `collections`). Nada que instalar.


## 1. Por qué el texto es un dato difícil

Una máquina no "entiende" palabras: solo opera con números. Y el lenguaje humano es endemoniadamente irregular. Mira estas tres formas de nombrar lo mismo en compras públicas:

- `"Artículos de Aseo e Higiene"`
- `"ARTICULOS DE ASEO E HIGIENE"`
- `"articulos aseo, higiene"`

Para ti son lo mismo. Para un computador, **son tres textos distintos**: cambian las mayúsculas, las tildes, los signos. Si quisieras contar cuántas veces aparece "aseo", el computador te diría que cada una es única. Por eso el NLP **siempre empieza limpiando**.

### La tubería básica del NLP

Casi todo proyecto de texto sigue estos pasos, de crudo a útil:

1. **Normalizar:** poner todo en minúsculas, quitar tildes y signos. Así `"Médicos"` y `"medicos"` pasan a ser la **misma** palabra.
2. **Tokenizar:** partir el texto en unidades —normalmente palabras— llamadas *tokens*. `"equipos de oficina"` → `["equipos", "de", "oficina"]`.
3. **Quitar palabras vacías (stopwords):** palabras que aparecen en todo y no aportan tema: *de, y, para, el, la…*. Si no las quitas, "de" sería la palabra más común de cualquier texto en español y no te diría nada.
4. **Representar:** convertir las palabras en números. La forma más simple es la **bolsa de palabras** (*bag of words*): contar cuántas veces aparece cada término, ignorando el orden.

Con esos cuatro pasos ya puedes responder preguntas reales: ¿qué compra más el Estado?, ¿qué dos categorías se parecen?, ¿a qué rubro corresponde esta solicitud? Eso haremos.

> **Una analogía:** normalizar y tokenizar es como **ordenar la bodega** antes de hacer el inventario. Si los mismos productos están con etiquetas distintas y revueltos, cualquier conteo te miente. Primero ordenas, después cuentas.


## 2. Preparación y datos

El **corpus** (el conjunto de textos con el que trabajamos) son las **categorías reales de compra del Estado** —los *rubros* del catálogo de ChileCompra. Es texto real, en español y "sucio" de verdad: tildes, mayúsculas y palabras vacías. Perfecto para practicar.

Ejecuta la celda: deja disponibles `CORPUS` (la lista de rubros) y `STOPWORDS` (palabras vacías del español que vamos a ignorar).


In [ ]:
import os
import urllib.request
import pandas as pd
import re
import unicodedata
from collections import Counter

# Si el archivo no existe localmente (ej: en Colab), intentamos descargarlo desde GitHub
if not os.path.exists("rubros.csv"):
    try:
        url = "https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/C1-introduccion-al-nlp/rubros.csv"  # Reemplazar por la URL real del repositorio al publicar
        urllib.request.urlretrieve(url, "rubros.csv")
        print("rubros.csv descargado automáticamente.")
    except Exception:
        print("Aviso: No se pudo descargar rubros.csv automáticamente.")

# Carga de datos desde el archivo estático
df_rubros = pd.read_csv("rubros.csv")

# Corpus real: rubros del catálogo de compras públicas (ChileCompra). Traído por la MCP.
# Fuente oficial: Dirección de Compras y Contratación Pública (ChileCompra / MercadoPúblico)
CORPUS = df_rubros["rubro"].tolist()

# Palabras vacías del español (no aportan tema). Lista provista para todo el módulo.
STOPWORDS = {
    "de", "y", "e", "o", "u", "para", "el", "la", "los", "las", "un", "una",
    "en", "con", "a", "del", "al", "por", "su", "sus", "se", "que", "general",
}

print(f"Corpus: {len(CORPUS)} rubros reales del Estado.")
print("Ejemplo:", CORPUS[12])


## Ejercicio 1 — Normalizar el texto

El primer paso es dejar todos los textos en una forma **uniforme**, para que `"Médicos"`, `"MEDICOS"` y `"medicos"` sean la misma palabra.

Programa la función `normalizar(texto)` que haga, en orden:

1. Pasar todo a **minúsculas** (`.lower()`).
2. **Quitar las tildes**. El truco estándar: descomponer los caracteres con `unicodedata.normalize("NFKD", texto)` y luego descartar las marcas de acento (`unicodedata.combining(c)`).
3. **Reemplazar** todo lo que no sea letra o número por un espacio (con `re.sub`).
4. **Colapsar** espacios múltiples en uno y recortar los extremos (`.strip()`).

> 💡 **Pista para quitar tildes:**
> ```python
> texto = unicodedata.normalize("NFKD", texto)
> texto = "".join(c for c in texto if not unicodedata.combining(c))
> ```

> ⚠️ **Error típico:** quitar los signos *antes* de bajar a minúsculas y descomponer tildes. Sigue el orden: minúsculas → tildes → signos → espacios.


In [ ]:
def normalizar(texto):
    texto = texto.lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = re.sub(r"[^a-z0-9\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

print(normalizar("Artículos ELÉCTRICOS, y de Iluminación!"))

In [ ]:
# Celda de chequeo — Ejercicio 1
def _chequeo_1():
    try:
        f = normalizar
    except NameError:
        print("❌ Aún no defines la función `normalizar`."); return
    casos = {
        "Artículos ELÉCTRICOS, y de Iluminación!": "articulos electricos y de iluminacion",
        "Equipamiento y SUMINISTROS médicos": "equipamiento y suministros medicos",
        "Á É Í Ó Ú ñ 2025!!": "a e i o u n 2025",
    }
    try:
        for entrada, esperado in casos.items():
            obtenido = f(entrada)
            assert obtenido == esperado, f"con «{entrada}» esperaba «{esperado}» y obtuve «{obtenido}»"
        print("✅ Correcto: normalización uniforme (minúsculas, sin tildes ni signos).")
    except AssertionError as e:
        print(f"❌ Casi. Pista: {e}")
    except Exception as e:
        print(f"❌ La función falló: {e}. Revisa los pasos uno por uno.")

_chequeo_1()

## Ejercicio 2 — Tokenizar y quitar palabras vacías

Ya tenemos texto limpio. Ahora lo **partimos en palabras** (tokens) y descartamos las **palabras vacías** (`STOPWORDS`), que aparecen en todo y no aportan tema.

Programa la función `tokenizar(texto)` que:

1. Normalice el texto (reutiliza tu `normalizar`).
2. Lo parta por espacios (`.split()`).
3. Devuelva solo los tokens que **no** estén en `STOPWORDS` **y** tengan **3 o más letras** (así se cuelan menos restos sin valor).

> 💡 **Pista:** una comprensión de lista lo resuelve en una línea:
> `[t for t in normalizar(texto).split() if t not in STOPWORDS and len(t) >= 3]`

> ⚠️ **Error típico:** olvidar reutilizar `normalizar`. Si tokenizas el texto crudo, "Médicos!" y "medicos" quedarán como tokens distintos.


In [ ]:
def tokenizar(texto):
    return [t for t in normalizar(texto).split()
            if t not in STOPWORDS and len(t) >= 3]

print(tokenizar("Servicios de Limpieza Industrial"))

In [ ]:
# Celda de chequeo — Ejercicio 2
def _chequeo_2():
    try:
        f = tokenizar
    except NameError:
        print("❌ Aún no defines la función `tokenizar`."); return
    try:
        assert f("Servicios de Limpieza Industrial") == ["servicios", "limpieza", "industrial"], \
            "«Servicios de Limpieza Industrial» debería dar ['servicios','limpieza','industrial']."
        assert f("Equipamiento y suministros médicos") == ["equipamiento", "suministros", "medicos"], \
            "Revisa que quites 'y' (stopword) y normalices las tildes."
        assert "de" not in f("Productos de papel") and f("Productos de papel") == ["productos", "papel"], \
            "La palabra vacía 'de' no debería sobrevivir."
        print("✅ Correcto: tokens limpios, sin palabras vacías ni tildes.")
    except AssertionError as e:
        print(f"❌ Casi. Pista: {e}")
    except Exception as e:
        print(f"❌ La función falló: {e}. ¿Reutilizaste `normalizar`?")

_chequeo_2()

## Ejercicio 3 — Bolsa de palabras: ¿qué compra más el Estado?

Con el texto tokenizado podemos **contar**. La idea de la **bolsa de palabras** es simple: juntar todos los tokens de todo el corpus y contar cuántas veces aparece cada uno. Las palabras más frecuentes nos dicen de qué trata, en conjunto, lo que compra el Estado.

`collections.Counter` hace el conteo por ti.

Tu tarea:

1. Recorre `CORPUS`, tokeniza cada rubro y **acumula** todos los tokens en un único `Counter` llamado `frecuencias`.
2. Mira los más comunes con `frecuencias.most_common(8)`.

> 💡 **Pista:** crea `frecuencias = Counter()` y dentro del bucle haz `frecuencias.update(tokenizar(rubro))`.

> ⚠️ **Error típico:** contar los rubros como frases completas en vez de por palabra. Queremos contar **tokens**, no textos enteros.


In [ ]:
frecuencias = Counter()
for rubro in CORPUS:
    frecuencias.update(tokenizar(rubro))

print("Top 8 palabras del catálogo de compras:")
for palabra, n in frecuencias.most_common(8):
    print(f"  {palabra:14s} {n}")

In [ ]:
# Celda de chequeo — Ejercicio 3
def _chequeo_3():
    # Recomputamos la verdad de forma independiente
    def _norm(t):
        t = t.lower(); t = unicodedata.normalize("NFKD", t)
        t = "".join(c for c in t if not unicodedata.combining(c))
        t = re.sub(r"[^a-z0-9\s]", " ", t)
        return re.sub(r"\s+", " ", t).strip()
    def _tok(t):
        return [w for w in _norm(t).split() if w not in STOPWORDS and len(w) >= 3]
    esperado = Counter()
    for r in CORPUS:
        esperado.update(_tok(r))
    try:
        fr = frecuencias
    except NameError:
        print("❌ Aún no existe `frecuencias`. Completa el TODO."); return
    if fr is None:
        print("❌ `frecuencias` sigue en None."); return
    try:
        assert isinstance(fr, Counter), "`frecuencias` debe ser un Counter."
        top_esp = esperado.most_common(1)[0]
        assert fr.most_common(1)[0] == top_esp, \
            f"La palabra más común debería ser '{top_esp[0]}' ({top_esp[1]} veces)."
        for w in ["servicios", "suministros", "productos"]:
            assert fr[w] == esperado[w], f"El conteo de '{w}' no coincide ({fr[w]} vs {esperado[w]})."
        print("✅ Correcto: bolsa de palabras construida.")
        print(f"   Palabra más frecuente: '{top_esp[0]}' ({top_esp[1]} veces).")
        print("   El Estado compra, sobre todo:",
              ", ".join(w for w, _ in esperado.most_common(5)))
    except AssertionError as e:
        print(f"❌ Casi. Pista: {e}")

_chequeo_3()

## Ejercicio 4 — Un clasificador simple: ¿a qué rubro va mi compra?

Imagina la pantalla de un funcionario que escribe en lenguaje natural lo que necesita comprar, y el sistema le sugiere el **rubro** correcto. Esa es una tarea clásica de NLP: **clasificar texto**. Vamos a construir la versión más simple y honesta: **por palabras en común**.

La idea: tomamos el texto de la solicitud, lo tokenizamos, y lo comparamos con cada rubro. El rubro que **comparte más palabras** con la solicitud gana. Es una **bolsa de palabras** aplicada a la similitud.

Programa la función `clasificar(texto)` que:

1. Tokenice `texto` y lo guarde como un **conjunto** (`set`) de palabras.
2. Para cada rubro de `CORPUS`, cuente **cuántas palabras comparte** con la solicitud (intersección de conjuntos).
3. Devuelva el **rubro con mayor coincidencia**.

> 💡 **Pista:** `max(CORPUS, key=lambda r: len(consulta & set(tokenizar(r))))` te da el rubro ganador en una línea, donde `consulta` es el set de tokens del texto.

> ⚠️ **Límite honesto:** este clasificador solo acierta si la solicitud usa palabras **parecidas** a las del nombre del rubro. No entiende sinónimos ni contexto (no sabe que "jeringa" es algo médico). Resolver eso es justo lo que traen los modelos de lenguaje de D16.


In [ ]:
def clasificar(texto):
    consulta = set(tokenizar(texto))
    return max(CORPUS, key=lambda r: len(consulta & set(tokenizar(r))))

for s in ["Necesito comprar medicamentos y productos farmacéuticos",
          "servicios de limpieza para la oficina"]:
    print(f"«{s}»  →  {clasificar(s)}")

In [ ]:
# Celda de chequeo — Ejercicio 4
def _chequeo_4():
    try:
        f = clasificar
    except NameError:
        print("❌ Aún no defines la función `clasificar`."); return
    pruebas = {
        "Necesito comprar medicamentos y productos farmacéuticos":
            "Medicamentos y productos farmacéuticos",
        "servicios de limpieza para la oficina":
            "Servicios de limpieza industrial",
    }
    try:
        for entrada, esperado in pruebas.items():
            obtenido = f(entrada)
            assert obtenido == esperado, \
                f"«{entrada}» debería clasificarse como «{esperado}», no «{obtenido}»."
        print("✅ Correcto: clasificador por palabras clave funcionando.")
        print("   «medicamentos…» →", f("Necesito comprar medicamentos y productos farmacéuticos"))
    except AssertionError as e:
        print(f"❌ Casi. Pista: {e}")
    except Exception as e:
        print(f"❌ La función falló: {e}. ¿Tokenizaste y usaste intersección de conjuntos?")

_chequeo_4()

## Cierre — Lo que te llevas

- El texto es un dato **difícil** porque es irregular; por eso el NLP siempre empieza **limpiando**.
- La tubería básica: **normalizar → tokenizar → quitar stopwords → representar** (bolsa de palabras).
- Con conteos simples ya respondes preguntas reales (qué se compra más) y construyes un **clasificador por palabras clave**.
- Y viste el **límite**: contar palabras no captura significado ni sinónimos. Ese salto —entender el texto, no solo contarlo— es lo que traen los **modelos de lenguaje (LLMs)**, el tema de **D16**.

### Para llevar al trabajo
Cada vez que tengas una columna de texto libre —descripciones, observaciones, reclamos, glosas— ya tienes el primer kit: límpiala, cuéntala y úsala para agrupar o clasificar. Es asombroso cuánto se entiende de un montón de texto solo **contando bien las palabras correctas**.

---
*Formación Pública · Contenido bajo licencia CC BY 4.0. Datos: catálogo de rubros de ChileCompra / MercadoPúblico.*
